In [12]:
from pathlib import Path
import pandas as pd
import plotly.express as px

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "processed" / "netflix_cleaned_dataset.csv"
df = pd.read_csv(data_path)

rating_summary = (
    df["rating"]
    .value_counts(dropna=False)
    .rename_axis("rating")
    .to_frame("title_count")
)

rating_summary["percentage"] = (
    rating_summary["title_count"] / len(df) * 100
).round(2)

display(rating_summary)

,title_count,percentage
rating,,
TV-MA,3205,36.46
TV-14,2157,24.54
TV-PG,861,9.80
R,799,9.09
PG-13,490,5.57
TV-Y7,333,3.79
TV-Y,306,3.48
PG,287,3.27
TV-G,220,2.50


In [13]:
genre_titles = (
    df[["show_id", "listed_in"]]
    .dropna(subset=["listed_in"])
    .assign(genre=lambda data: data["listed_in"].str.split(r"\s*,\s*"))
    .explode("genre")
)

genre_titles["genre"] = genre_titles["genre"].str.strip()
genre_titles = (
    genre_titles[genre_titles["genre"].ne("")]
    .drop_duplicates(subset=["show_id", "genre"])
)

genre_counts = (
    genre_titles.groupby("genre")["show_id"]
    .nunique()
    .sort_values(ascending=False)
)

display(genre_counts.head(15).rename("title_count").to_frame())

,title_count
genre,
International Movies,2752
Dramas,2426
Comedies,1674
International TV Shows,1349
Documentaries,869
Action & Adventure,859
TV Dramas,762
Independent Movies,756
Children & Family Movies,641


In [14]:
rating_by_type = pd.crosstab(df["rating"], df["type"])

rating_pct_by_type = (
    rating_by_type
    .div(rating_by_type.sum(axis=0), axis=1)
    .mul(100)
    .round(1)
)

display(rating_pct_by_type)

type,Movie,TV Show
rating,,
G,0.7,0.0
NC-17,0.0,0.0
NR,1.2,0.2
PG,4.7,0.0
PG-13,8.0,0.0
R,13.0,0.1
TV-14,23.3,27.4
TV-G,2.1,3.5
TV-MA,33.7,42.9


In [15]:
# interactive visualizations
top_genres = (
    genre_counts.head(15)
    .rename_axis("genre")
    .reset_index(name="title_count")
)

fig_genres = px.bar(
    top_genres.sort_values("title_count"),
    x = "title_count",
    y = "genre",
    orientation = "h",
    title = "Top 15 Netflix Genres",
    labels = {
        "title_count": "Number of titles",
        "genre": "Genre"
    },
    color = "title_count",
    color_continuous_scale = "Reds",
    text="title_count"
)

fig_genres.update_traces(textposition = "outside")
fig_genres.update_layout(yaxis = {"categoryorder": "total ascending"})
fig_genres.show()

In [16]:
rating_plot_data = (
    rating_pct_by_type
    .rename_axis("rating")
    .reset_index()
    .melt(
        id_vars = "rating",
        var_name = "content_type",
        value_name = "percentage"
    )
)

fig_ratings = px.bar(
    rating_plot_data,
    x = "rating",
    y = "percentage",
    color = "content_type",
    barmode = "group",
    title = "Rating Distribution by Content Type",
    labels = {
        "rating": "Content rating",
        "percentage": "Share within content type (%)",
        "content_type": "Content type"
    },
    hover_data = {"percentage": ":.1f"}
)
fig_ratings.show()

# Audience insights

- **Ratings:** TV-MA is the most common rating for both Movies and TV Shows in this dataset, representing 33.7% of Movie titles and 42.9% of TV Shows.
- **Family-oriented ratings:** TV-Y and TV-Y7 combined make up 13.9% of TV Shows, compared with 4.4% of Movies.
- **Common genres:** The most common genre labels are International Movies, Dramas, and Comedies.

These patterns describe how titles in this catalog are classified; ratings indicate suitability categories, not the actual ages or preferences of viewers. Genre labels can overlap because titles may have multiple genres.